### Configuração e Conexão com o Banco

In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import os

# configurações
DB_HOST = 'localhost' 
DB_PORT = '5432'
DB_NAME = 'ceap_dw'
DB_USER = 'admin'
DB_PASS = 'admin_password'

# Caminhos dos Arquivos
RAW_PATH_MAIN = '../data layer/raw/deputies_dataset.csv'
RAW_PATH_V2 = '../data layer/raw/dirty_deputies_v2.csv'

# Conexão Banco
db_url = f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

print(" 1. Bibliotecas importadas e configuração de banco pronta.")

 1. Bibliotecas importadas e configuração de banco pronta.


### EXTRACT (Extração)

In [3]:
print(" Iniciando leitura dos arquivos RAW...")

try:
    #cria dois dataframes sendo: df_raw e df_v2
    df_raw = pd.read_csv(RAW_PATH_MAIN, low_memory=False) #força leitura consistente de tipos
    print(f"   - Arquivo Principal carregado: {len(df_raw)} linhas.")

    df_v2 = pd.read_csv(RAW_PATH_V2, low_memory=False)
    print(f"   - Arquivo V2 (Enriquecimento) carregado: {len(df_v2)} linhas.")

except Exception as e:
    print(f" Erro na leitura dos arquivos: {e}")

 Iniciando leitura dos arquivos RAW...
   - Arquivo Principal carregado: 3014902 linhas.
   - Arquivo V2 (Enriquecimento) carregado: 339089 linhas.


### TRANSFORM (Limpeza e Padronização)

In [4]:
print("Iniciando tratamento de dados...")

#  Filtra bugged_date
# Se existir a coluna 'bugged_date', usamos ela como gate de qualidade.
if 'bugged_date' in df_raw.columns:
    df_silver = df_raw[df_raw['bugged_date'] == 0].copy()
    df_silver = df_silver.drop(columns=['bugged_date'], errors='ignore')
else:
    df_silver = df_raw.copy()

#  Tipagem Numérica e Data
#Converte valores monetários para numérico e Qualquer valor inválido vira NaN
df_silver['receipt_value'] = pd.to_numeric(df_silver['receipt_value'], errors='coerce')
df_silver['receipt_date'] = pd.to_datetime(df_silver['receipt_date'], errors='coerce')

#  Padronização de Texto Geral e Lista de colunas que precisam ser normalizadas
text_cols = ['deputy_name', 'political_party', 'establishment_name', 'receipt_description', 'receipt_social_security_number']
for col in text_cols:
    if col in df_silver.columns:
        df_silver[col] = df_silver[col].astype(str).str.upper().str.strip()
        

# Unifica o nome da coluna para evitar lógica duplicada no pipeline
if 'deputy_state' in df_silver.columns:
    df_silver = df_silver.rename(columns={'deputy_state': 'state_code'})

if 'state_code' in df_silver.columns:
    df_silver['state_code'] = df_silver['state_code'].astype(str).str.upper().str.strip()
    
    #  Tratamento de "nan" (pandas transforma nulo em texto "nan")
    # Se for "NAN", vira string vazia ou NaN
    df_silver.loc[df_silver['state_code'] == 'NAN', 'state_code'] = None
    
    df_silver['state_code'] = df_silver['state_code'].str.slice(0, 2)
    
    print("   - Coluna 'state_code' tratada e truncada para 2 caracteres.")

print(f" 3. Limpeza concluída. Linhas prontas: {len(df_silver)}")

Iniciando tratamento de dados...
   - Coluna 'state_code' tratada e truncada para 2 caracteres.
 3. Limpeza concluída. Linhas prontas: 2965371


### ENRICH (Enriquecimento com Join - Juntando os CSV)

In [5]:

print(" Cruzando dados com a tabela V2 (Partidos)...")

# Valida se a chave de junção existe na base V2
if 'political_party' in df_v2.columns:
    # Renomear ideologia 
    if 'party_ideology1' in df_v2.columns:
        df_v2 = df_v2.rename(columns={'party_ideology1': 'party_ideology'})

    # Padroniza a chave e Evita falha de merge por divergência de texto
    df_v2['political_party'] = df_v2['political_party'].astype(str).str.upper().str.strip()
    
    # Tratamento de datas relevantes da dimensão partidária
    if 'party_regdate' in df_v2.columns:
        df_v2['party_regdate'] = pd.to_datetime(df_v2['party_regdate'], dayfirst=True, errors='coerce')
        print("   - Datas de fundação dos partidos convertidas com sucesso.")

    # Remove duplicatas na V2
    df_v2_unique = df_v2.drop_duplicates(subset=['political_party'])

    # Faz o Join
    cols_v2 = ['political_party', 'party_regdate', 'party_ideology']
    cols_final_v2 = [c for c in cols_v2 if c in df_v2_unique.columns]
    
    # Atualiza o df_silver com as novas colunas
    df_silver = pd.merge(df_silver, df_v2_unique[cols_final_v2], on='political_party', how='left')
    
    print(" 4. Join realizado! Dados enriquecidos.")
else:
    print(" Aviso: Coluna 'political_party' não encontrada na V2.")

 Cruzando dados com a tabela V2 (Partidos)...


C:\Users\lucas\AppData\Local\Temp\ipykernel_72048\4280263995.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_v2['party_regdate'] = pd.to_datetime(df_v2['party_regdate'], dayfirst=True, errors='coerce')


   - Datas de fundação dos partidos convertidas com sucesso.
 4. Join realizado! Dados enriquecidos.


### LOAD (Carregar no Banco)

In [6]:
# Load - Popula o banco com os dados do df_silver
from sqlalchemy import text

# Define destino da carga
target_table = 'tb_reembolso'
target_schema = 'silver'

print(f"Inserindo dados na tabela '{target_schema}.{target_table}'...")

try:
     # Abre conexão explícita com o banco
    with engine.connect() as conn:
        # Limpa os dados, mas mantém a estrutura do DDL
        conn.execute(text(f"TRUNCATE TABLE {target_schema}.{target_table} RESTART IDENTITY;"))
        conn.commit()
        print("   - Tabela limpa (Truncate realizado).")

    # Insere os dados do DataFrame no banco (append)
    df_silver.to_sql(
        target_table,
        engine,
        schema=target_schema,
        if_exists='append', 
        index=False
    )

    print(f" SUCESSO! Dados inseridos na tabela Silver.")

except Exception as e:
    # Captura falhas de conexão, permissão, schema ou tipo de dado
    print(f"Erro: {e}")

Inserindo dados na tabela 'silver.tb_reembolso'...
   - Tabela limpa (Truncate realizado).
 SUCESSO! Dados inseridos na tabela Silver.
